# Signal Processing & Spectral Analysis

**Category:** 10-Scientific  
**Description:** FFT, filtering, and frequency domain analysis with AI assistance  
**Libraries:** NumPy, SciPy, Matplotlib

---

In [ ]:
# =============================================================================
# DEPENDENCIES & MODEL ALIASES
# =============================================================================
%pip install -q numpy scipy matplotlib

# Model aliases - update when models change
CLAUDE = "anthropic-chat:claude-sonnet-4-5-20250929"
CLAUDE_FAST = "anthropic-chat:claude-haiku-4-5-20251001"
GPT = "openai-chat:gpt-5"
GEMINI = "gemini:gemini-2.5-flash"

MODEL = CLAUDE  # Default model for this notebook

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal, fft
from scipy.io import wavfile

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True

---

# Part 1: Signal Generation

---

In [ ]:
# Generate a composite signal with multiple frequencies
fs = 1000  # Sampling frequency (Hz)
t = np.linspace(0, 2, 2 * fs)  # 2 seconds of data

# Signal components
f1, f2, f3 = 5, 20, 50  # Frequencies in Hz
signal_clean = (
    1.0 * np.sin(2 * np.pi * f1 * t) +  # 5 Hz component
    0.5 * np.sin(2 * np.pi * f2 * t) +  # 20 Hz component
    0.3 * np.sin(2 * np.pi * f3 * t)    # 50 Hz component
)

# Add noise
np.random.seed(42)
noise = 0.5 * np.random.randn(len(t))
signal_noisy = signal_clean + noise

# Plot
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

axes[0].plot(t[:500], signal_clean[:500], 'b-', linewidth=1)
axes[0].set_title(f'Clean Signal: {f1}Hz + {f2}Hz + {f3}Hz')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')

axes[1].plot(t[:500], signal_noisy[:500], 'r-', linewidth=0.5, alpha=0.7)
axes[1].set_title('Noisy Signal')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Amplitude')

plt.tight_layout()
plt.show()

---

# Part 2: Fourier Transform & Spectral Analysis

---

In [ ]:
def compute_spectrum(sig, fs):
    """Compute single-sided amplitude spectrum."""
    n = len(sig)
    yf = fft.fft(sig)
    xf = fft.fftfreq(n, 1/fs)[:n//2]
    amplitude = 2.0/n * np.abs(yf[:n//2])
    return xf, amplitude

# Compute spectra
freq_clean, amp_clean = compute_spectrum(signal_clean, fs)
freq_noisy, amp_noisy = compute_spectrum(signal_noisy, fs)

# Plot frequency domain
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].stem(freq_clean[:100], amp_clean[:100], 'b', markerfmt='bo', basefmt=' ')
axes[0].set_title('Clean Signal Spectrum')
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('Amplitude')
axes[0].set_xlim(0, 60)

axes[1].plot(freq_noisy[:200], amp_noisy[:200], 'r-', alpha=0.7)
axes[1].set_title('Noisy Signal Spectrum')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Amplitude')
axes[1].set_xlim(0, 100)

plt.tight_layout()
plt.show()

# Find peaks
peaks, _ = signal.find_peaks(amp_clean, height=0.1)
print(f"Detected frequency components: {freq_clean[peaks]} Hz")

In [ ]:
%%ai $MODEL
Explain the Fourier Transform in simple terms:
1. What does FFT actually compute?
2. Why do we see peaks at 5Hz, 20Hz, and 50Hz in the spectrum?
3. What's the relationship between sampling frequency and the max detectable frequency?

Keep the explanation accessible to someone with basic math knowledge.

---

# Part 3: Digital Filtering

---

In [ ]:
# Design filters
def design_filter(filter_type, cutoff, fs, order=4):
    """Design a Butterworth filter."""
    nyq = 0.5 * fs
    normalized_cutoff = np.array(cutoff) / nyq
    b, a = signal.butter(order, normalized_cutoff, btype=filter_type)
    return b, a

# Low-pass filter (remove 50Hz, keep 5Hz and 20Hz)
b_low, a_low = design_filter('low', 30, fs)
filtered_low = signal.filtfilt(b_low, a_low, signal_noisy)

# Band-pass filter (keep only 15-25Hz, isolate the 20Hz component)
b_band, a_band = design_filter('band', [15, 25], fs)
filtered_band = signal.filtfilt(b_band, a_band, signal_noisy)

# High-pass filter (remove 5Hz, keep 20Hz and 50Hz)
b_high, a_high = design_filter('high', 10, fs)
filtered_high = signal.filtfilt(b_high, a_high, signal_noisy)

In [ ]:
# Plot filter results
fig, axes = plt.subplots(4, 1, figsize=(14, 10))

# Original noisy
axes[0].plot(t[:500], signal_noisy[:500], 'gray', linewidth=0.5, alpha=0.5, label='Noisy')
axes[0].plot(t[:500], signal_clean[:500], 'b-', linewidth=1, label='Original')
axes[0].set_title('Original Signal')
axes[0].legend()

# Low-pass
axes[1].plot(t[:500], filtered_low[:500], 'g-', linewidth=1)
axes[1].set_title('Low-Pass Filtered (cutoff=30Hz) - Removes 50Hz')

# Band-pass
axes[2].plot(t[:500], filtered_band[:500], 'orange', linewidth=1)
axes[2].set_title('Band-Pass Filtered (15-25Hz) - Isolates 20Hz component')

# High-pass
axes[3].plot(t[:500], filtered_high[:500], 'purple', linewidth=1)
axes[3].set_title('High-Pass Filtered (cutoff=10Hz) - Removes 5Hz')

for ax in axes:
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude')

plt.tight_layout()
plt.show()

In [ ]:
# Filter frequency response
fig, ax = plt.subplots(figsize=(10, 5))

for (b, a), label, color in [
    ((b_low, a_low), 'Low-pass (30Hz)', 'green'),
    ((b_band, a_band), 'Band-pass (15-25Hz)', 'orange'),
    ((b_high, a_high), 'High-pass (10Hz)', 'purple')
]:
    w, h = signal.freqz(b, a, worN=2000)
    freq = w * fs / (2 * np.pi)
    ax.plot(freq, 20 * np.log10(abs(h)), color, label=label, linewidth=2)

ax.set_title('Filter Frequency Response')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Gain (dB)')
ax.set_xlim(0, 100)
ax.set_ylim(-60, 5)
ax.axhline(-3, color='gray', linestyle='--', label='-3dB (cutoff)')
ax.legend()
ax.grid(True)
plt.show()

---

# Part 4: Spectrogram Analysis

---

In [ ]:
# Create a chirp signal (frequency sweep)
t_chirp = np.linspace(0, 5, 5 * fs)
chirp_signal = signal.chirp(t_chirp, f0=1, f1=100, t1=5, method='linear')

# Add some noise
chirp_noisy = chirp_signal + 0.2 * np.random.randn(len(t_chirp))

# Compute spectrogram
f, t_spec, Sxx = signal.spectrogram(chirp_noisy, fs, nperseg=256)

# Plot
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(t_chirp, chirp_noisy, 'b-', linewidth=0.3)
axes[0].set_title('Chirp Signal (Linear Frequency Sweep 1-100Hz)')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')

im = axes[1].pcolormesh(t_spec, f, 10 * np.log10(Sxx), shading='gouraud', cmap='viridis')
axes[1].set_title('Spectrogram')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Frequency (Hz)')
axes[1].set_ylim(0, 120)
plt.colorbar(im, ax=axes[1], label='Power (dB)')

plt.tight_layout()
plt.show()

In [ ]:
%%ai $MODEL
Looking at the chirp signal and its spectrogram:

1. What applications use chirp signals? (radar, sonar, etc.)
2. Why is the spectrogram useful for analyzing non-stationary signals?
3. What's the trade-off between time resolution and frequency resolution in spectrograms?

---

# Part 5: Real-World Example - ECG Signal Analysis

---

In [ ]:
# Simulate an ECG-like signal
def generate_ecg(duration, fs, heart_rate=72):
    """Generate a synthetic ECG signal."""
    t = np.linspace(0, duration, int(duration * fs))
    
    # Period of one heartbeat
    period = 60 / heart_rate
    
    # Generate QRS complex pattern
    ecg = np.zeros_like(t)
    for beat_time in np.arange(0, duration, period):
        # P wave
        ecg += 0.15 * np.exp(-((t - beat_time - 0.1) ** 2) / 0.002)
        # QRS complex
        ecg -= 0.1 * np.exp(-((t - beat_time - 0.18) ** 2) / 0.0002)
        ecg += 1.0 * np.exp(-((t - beat_time - 0.2) ** 2) / 0.0003)
        ecg -= 0.15 * np.exp(-((t - beat_time - 0.22) ** 2) / 0.0002)
        # T wave
        ecg += 0.2 * np.exp(-((t - beat_time - 0.35) ** 2) / 0.005)
    
    return t, ecg

# Generate ECG
ecg_fs = 500  # 500 Hz sampling
t_ecg, ecg_clean = generate_ecg(5, ecg_fs, heart_rate=72)

# Add realistic noise (baseline wander + muscle noise + powerline interference)
baseline_wander = 0.1 * np.sin(2 * np.pi * 0.3 * t_ecg)  # 0.3 Hz
muscle_noise = 0.05 * np.random.randn(len(t_ecg))
powerline = 0.08 * np.sin(2 * np.pi * 60 * t_ecg)  # 60 Hz
ecg_noisy = ecg_clean + baseline_wander + muscle_noise + powerline

# Plot
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

axes[0].plot(t_ecg, ecg_clean, 'b-', linewidth=1)
axes[0].set_title('Clean ECG Signal (72 BPM)')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('mV')

axes[1].plot(t_ecg, ecg_noisy, 'r-', linewidth=0.5)
axes[1].set_title('Noisy ECG (baseline wander + muscle noise + 60Hz powerline)')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('mV')

plt.tight_layout()
plt.show()

In [ ]:
# ECG Filtering pipeline

# 1. High-pass filter to remove baseline wander (cutoff 0.5Hz)
b_hp, a_hp = design_filter('high', 0.5, ecg_fs, order=2)
ecg_hp = signal.filtfilt(b_hp, a_hp, ecg_noisy)

# 2. Notch filter to remove 60Hz powerline interference
b_notch, a_notch = signal.iirnotch(60, 30, ecg_fs)
ecg_notch = signal.filtfilt(b_notch, a_notch, ecg_hp)

# 3. Low-pass filter to remove high-frequency noise (cutoff 40Hz)
b_lp, a_lp = design_filter('low', 40, ecg_fs, order=4)
ecg_filtered = signal.filtfilt(b_lp, a_lp, ecg_notch)

# Plot results
fig, axes = plt.subplots(4, 1, figsize=(14, 10))

axes[0].plot(t_ecg, ecg_noisy, 'gray', linewidth=0.5)
axes[0].set_title('1. Original Noisy ECG')

axes[1].plot(t_ecg, ecg_hp, 'orange', linewidth=0.8)
axes[1].set_title('2. After High-Pass (0.5Hz) - Baseline Wander Removed')

axes[2].plot(t_ecg, ecg_notch, 'purple', linewidth=0.8)
axes[2].set_title('3. After Notch Filter (60Hz) - Powerline Interference Removed')

axes[3].plot(t_ecg, ecg_filtered, 'green', linewidth=1)
axes[3].plot(t_ecg, ecg_clean, 'b--', linewidth=1, alpha=0.5, label='Ground Truth')
axes[3].set_title('4. Final Filtered ECG (Low-Pass 40Hz)')
axes[3].legend()

for ax in axes:
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('mV')

plt.tight_layout()
plt.show()

In [ ]:
# R-peak detection for heart rate calculation
peaks, properties = signal.find_peaks(ecg_filtered, height=0.5, distance=ecg_fs*0.5)

# Calculate heart rate from R-R intervals
rr_intervals = np.diff(t_ecg[peaks])  # Time between peaks
heart_rates = 60 / rr_intervals  # Convert to BPM

print(f"Detected {len(peaks)} R-peaks")
print(f"Average Heart Rate: {np.mean(heart_rates):.1f} BPM")
print(f"Heart Rate Variability (std): {np.std(heart_rates):.2f} BPM")

# Plot with detected peaks
plt.figure(figsize=(14, 4))
plt.plot(t_ecg, ecg_filtered, 'b-', linewidth=1)
plt.plot(t_ecg[peaks], ecg_filtered[peaks], 'ro', markersize=8, label='R-peaks')
plt.title(f'ECG with Detected R-peaks (HR: {np.mean(heart_rates):.1f} BPM)')
plt.xlabel('Time (s)')
plt.ylabel('mV')
plt.legend()
plt.show()

In [ ]:
%%ai $MODEL
For ECG signal processing:

1. Why is the order of filtering important (high-pass -> notch -> low-pass)?
2. What's the clinical significance of Heart Rate Variability (HRV)?
3. What other features can we extract from ECG for diagnostics?

---

# Part 6: Power Spectral Density

---

In [ ]:
# Compute Power Spectral Density using Welch's method
f_psd, psd_noisy = signal.welch(signal_noisy, fs, nperseg=256)
f_psd, psd_clean = signal.welch(signal_clean, fs, nperseg=256)
f_psd, psd_filtered = signal.welch(filtered_low, fs, nperseg=256)

fig, ax = plt.subplots(figsize=(12, 5))

ax.semilogy(f_psd, psd_noisy, 'r-', alpha=0.5, label='Noisy Signal', linewidth=1)
ax.semilogy(f_psd, psd_clean, 'b-', label='Clean Signal', linewidth=2)
ax.semilogy(f_psd, psd_filtered, 'g--', label='Filtered Signal', linewidth=2)

ax.set_title('Power Spectral Density (Welch Method)')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PSD (V²/Hz)')
ax.set_xlim(0, 80)
ax.legend()
ax.grid(True, which='both', alpha=0.3)

# Mark the signal frequencies
for f in [5, 20, 50]:
    ax.axvline(f, color='gray', linestyle=':', alpha=0.5)

plt.show()

---

## Summary

This notebook demonstrated:
- Signal generation and visualization
- Fast Fourier Transform (FFT) for spectral analysis
- Digital filter design (low-pass, high-pass, band-pass, notch)
- Spectrogram analysis for time-varying signals
- Real-world ECG signal processing pipeline
- Power Spectral Density estimation

**Next:** Explore physics simulations in `02-physics-simulation.ipynb`

---